# Визуализация — мини-проект

Сквозной мини-проект на реальных данных. Повторяет ключевые темы урока.


In [ ]:
# Setup
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print
import warnings
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, ConfusionMatrixDisplay

np.random.seed(42)



## Постановка задачи `(intro)`


Сквозной EDA на **Titanic (OpenML)**: pandas built-in → seaborn → диагностика модели. Все графики — одна цепочка решений.


## Загрузка и быстрый обзор `(eda)`


In [ ]:
warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", font_scale=1.05)

raw = fetch_openml("titanic", version=1, as_frame=True, parser="auto")
df = raw.frame.copy()
df["Survived"] = df["survived"].astype(int)
df["Pclass"] = df["pclass"].astype(int)
df["Age"] = pd.to_numeric(df["age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["fare"], errors="coerce")
df["Sex"] = df["sex"].astype(str)
print(df.shape)
df[["Age", "Fare", "Survived"]].describe()


## EDA: распределения и категории `(viz)`


**Решение:** строим базовые графики распределений и категорий — hist, box, violin, countplot.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
df["Fare"].plot(kind="hist", bins=30, ax=axes[0, 0], alpha=0.8)
axes[0, 0].set_title("pandas hist Fare")
sns.boxplot(data=df, x="Pclass", y="Fare", ax=axes[0, 1])
sns.violinplot(data=df, x="Sex", y="Age", ax=axes[1, 0])
sns.countplot(data=df, x="Survived", ax=axes[1, 1])
plt.tight_layout()
plt.show()


## Связи и корреляции `(viz)`


**Решение:** heatmap корреляций и scatter для пар признаков.


In [ ]:
num = df[["Survived", "Pclass", "Age", "Fare"]].dropna()
sns.heatmap(num.corr(), annot=True, cmap="coolwarm", center=0, vmin=-1, vmax=1)
plt.title("Корреляции числовых признаков")
plt.tight_layout()
plt.show()
sns.scatterplot(data=num, x="Fare", y="Age", hue="Survived", alpha=0.7)
plt.show()


## Модель и диагностика `(example)`


**Решение:** обучаем `LogisticRegression` в Pipeline и визуализируем confusion matrix + ROC.


In [ ]:
X = df[["Pclass", "Age", "Fare"]]
y = df["Survived"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
final_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=500, random_state=42)),
])
final_pipe.fit(X_train, y_train)
proba = final_pipe.predict_proba(X_test)[:, 1]
pred = final_pipe.predict(X_test)
final_model = final_pipe.named_steps["model"]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=axes[0], cmap="Blues")
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1])
plt.tight_layout()
plt.show()


## Чек-лист мини-проекта `(summary)`


1. Один датасет — цепочка графиков от EDA до ROC/CM.
2. seaborn для статистики; pandas.plot для быстрого старта.
3. Bar/box — ось Y с нуля; heatmap — center=0.
4. После модели — confusion matrix и ROC на том же test-set.
5. Подписи осей и title на каждом графике.
